# 01 — Data Extraction & Profiling

**Dataset:** Inside Airbnb — New York City 2019 (`AB_NYC_2019.csv`)  
**Source:** [Inside Airbnb](http://insideairbnb.com/get-the-data/) | Kaggle mirror  
**Objective:** Load the raw dataset, validate its integrity, and produce a comprehensive structural profile to guide downstream cleaning decisions.  

> This notebook is **read-only** — no data is modified here. All transformations happen in `02_cleaning.ipynb`.

**Analytical coverage:**
- NumPy array construction, masking, and vectorized operations for initial profiling
- Pandas loading, `.info()`, `.describe()`, dtype inspection, and missing-value quantification
- Row-wise logic to flag anomalous records before any transformation
- Descriptive statistics and distribution checks as a baseline for EDA

## 1.1 Environment Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 60)

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'AB_NYC_2019.csv'


## 1.2 Load Raw Dataset

In [ ]:
df = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Raw dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns.")
df.head()

## 1.3 Schema & Data Types

In [ ]:
df.info(show_counts=True)

In [ ]:
dtype_summary = pd.DataFrame({
    'dtype'        : df.dtypes.astype(str),
    'non_null'     : df.notna().sum(),
    'null_count'   : df.isna().sum(),
    'null_pct'     : (df.isna().sum() / len(df) * 100).round(2),
    'unique_values': df.nunique(),
    'sample_value' : [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
}).sort_values('null_pct', ascending=False)

dtype_summary

**Findings:**
- `last_review` and `reviews_per_month` share **10,052 nulls (~20.6%)** — these are listings that have never been reviewed. 
- `name` and `host_name` have minor gaps. 
- Core numeric KPIs are non-null at the source level.

## 1.4 Categorical Distributions

In [ ]:
cat_cols = ['neighbourhood_group', 'room_type']
for col in cat_cols:
    print(f"\nDistribution for {col}:")
    counts  = df[col].value_counts()
    pcts    = df[col].value_counts(normalize=True).mul(100).round(2)
    summary = pd.concat([counts, pcts], axis=1, keys=['count', 'pct_%'])
    print(summary)

## 1.5 Numeric Descriptive Statistics

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
desc = df[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
desc['range'] = desc['max'] - desc['min']
desc['iqr']   = desc['75%'] - desc['25%']
desc

## 1.6 NumPy-Powered Anomaly Flags (Pre-Cleaning Snapshot)

In [ ]:
price_arr      = df['price'].to_numpy(dtype=np.float64)
min_nights_arr = df['minimum_nights'].to_numpy(dtype=np.float64)
reviews_arr    = df['number_of_reviews'].to_numpy(dtype=np.float64)
avail_arr      = df['availability_365'].to_numpy(dtype=np.float64)

mask_zero_price     = price_arr == 0
mask_price_outlier  = price_arr > 1_000
mask_nights_extreme = min_nights_arr > 365
mask_avail_zero     = avail_arr == 0
mask_no_review      = reviews_arr == 0

anomaly_report = {
    'Zero price listings (to be REMOVED)'       : int(mask_zero_price.sum()),
    'Price > $1,000 (to be capped at p99)'      : int(mask_price_outlier.sum()),
    'Minimum nights > 365 (to be capped at 365)': int(mask_nights_extreme.sum()),
    'Availability = 0 (ghost / blocked listings)': int(mask_avail_zero.sum()),
    'Zero reviews (last_review will stay NaT)'  : int(mask_no_review.sum()),
}

print("ANOMALY REPORT:")
for k, v in anomaly_report.items():
    print(f"{k:<45}: {v:,}")

In [ ]:
print("PRICE PERCENTILES ($):")
pcts = np.percentile(price_arr[price_arr > 0], [10, 25, 50, 75, 90, 95, 99])
for p, v in zip([10, 25, 50, 75, 90, 95, 99], pcts):
    print(f"{p}th Percentile: ${v:.2f}")

## 1.7 Neighbourhood & Host-Level Summaries

In [ ]:
borough_summary = (
    df.groupby('neighbourhood_group')
    .agg(
        listing_count       = ('id'                             , 'count'),
        avg_price           = ('price'                          , 'mean'),
        median_price        = ('price'                          , 'median'),
        avg_min_nights      = ('minimum_nights'                 , 'mean'),
        avg_availability    = ('availability_365'               , 'mean'),
        total_reviews       = ('number_of_reviews'              , 'sum'),
        multi_listing_hosts = ('calculated_host_listings_count' , lambda x: (x > 1).sum())
    )
    .sort_values('listing_count', ascending=False)
    .round(2)
)
borough_summary

In [ ]:
top_hosts = (
    df.groupby('host_id')
    .agg(
        host_name     = ('host_name'        , 'first'),
        listing_count = ('id'               , 'count'),
        avg_price     = ('price'            , 'mean'),
        total_reviews = ('number_of_reviews', 'sum'),
    )
    .sort_values('listing_count', ascending=False)
    .head(15)
    .round(2)
)
top_hosts

## 1.8 Duplicate & ID Integrity Check

In [ ]:
print(f"Total duplicated rows: {df.duplicated().sum()}")
print(f"Duplicated IDs: {df.duplicated(subset='id').sum()}")

lr_non_null = pd.to_datetime(df['last_review'], errors='coerce').dropna()
print(f"Earliest last_review: {lr_non_null.min()}")
print(f"Latest last_review: {lr_non_null.max()}")

## 1.9 Extraction Summary & Cleaning Roadmap

The table below defines every transformation that `02_cleaning.ipynb` will perform.

| Issue | Raw Count | Action in `02_cleaning.ipynb` | Rationale |
|---|---|---|---|
| `name` nulls | 16 | Fill → `'Unknown Listing'` | Required for dashboard tooltip display |
| `host_name` nulls | 21 | Fill → `'Unknown Host'` | Required for host-level grouping |
| `reviews_per_month` nulls | 10,052 | Fill → `0.0` | 0 reviews means 0 rev/month |
| `last_review` nulls | 10,052 | **Keep as `NaT`** | Avoids axis corruption in Tableau |
| `price` = 0 | 11 | **Remove** | Corrupts revenue KPIs |
| `price` > p99 | ~239 | Cap at 99th percentile | Removes extreme skew |
| `minimum_nights` > 365 | 14 | Cap at 365 | Platform data error |

> **Proceed to** `02_cleaning.ipynb` for the full transformation pipeline.